# Module 2c - From a Live Literature Search to a Dataset

Type a topic, pull the open-access papers for it, read their **full text**, and turn them into a clean table you can model. This first part does the harvest: **an OpenAlex filter you build on the website, pasted here, becomes a corpus of full-text PDFs downloaded and parsed automatically.**

**The live-demo flow**
1. On **openalex.org** search a topic and click filters until the page reads something like:
   *works where open access is (true) and year >= (2019) and title/abstract has (battery capacity mAh) and type is (article) and citation count >= (100)*
2. Copy the filter (the site shows the API query), paste it into the config cell below.
3. Run: the notebook pulls that corpus, downloads each open-access full-text PDF, and parses it.

No key and no login for any of this (OpenAlex is free; a courtesy email is optional).

## 0. Setup

In [1]:
import sys, subprocess
if "google.colab" in sys.modules:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "requests", "pymupdf"])
import requests, re, io, time
print("ready")

ready


## 1. Paste your OpenAlex filter

Build the filter on **openalex.org** (search + click the filters), then paste it below. You can paste **either** the raw filter string **or** the whole `api.openalex.org/works?...` URL the site gives you - the notebook takes both. The default below is the battery example.

In [2]:
# Paste the filter string OR the full OpenAlex URL you built on the website:
OPENALEX_FILTER = "open_access.is_oa:true,publication_year:>2018,title_and_abstract.search:battery capacity mAh,type:article,cited_by_count:>99"

N_PAPERS = 20                       # how many full-text papers to harvest (demo: 15-30)
MAILTO   = "ruiding@uchicago.edu"   # courtesy only, NOT a key; may be left ""

def parse_filter(s):
    """Accept a raw filter string, or an openalex URL, and return the filter string."""
    s = s.strip()
    if "openalex.org" in s and "filter=" in s:
        from urllib.parse import urlparse, parse_qs, unquote
        return unquote(parse_qs(urlparse(s).query).get("filter", [""])[0])
    return s

FILTER = parse_filter(OPENALEX_FILTER)
print("using filter:\n ", FILTER)

using filter:
  open_access.is_oa:true,publication_year:>2018,title_and_abstract.search:battery capacity mAh,type:article,cited_by_count:>99


## 2. Pull the corpus

Ask OpenAlex for works matching the filter, sorted by citations, and keep the ones that actually have a downloadable open-access PDF.

In [3]:
def get_corpus(filt, n, mail=""):
    base, out, cursor = "https://api.openalex.org/works", [], "*"
    while len(out) < n:
        params = {"filter": filt, "sort": "cited_by_count:desc", "per-page": 50, "cursor": cursor}
        if mail: params["mailto"] = mail
        j = requests.get(base, params=params, timeout=45).json()
        res = j.get("results", [])
        if not res: break
        for w in res:
            loc = w.get("best_oa_location") or {}
            if not loc.get("pdf_url"): continue          # keep only papers with a real full-text PDF
            out.append(dict(title=(w.get("title") or "")[:100], year=w.get("publication_year"),
                            venue=(loc.get("source") or {}).get("display_name"),
                            cited=w.get("cited_by_count"), doi=w.get("doi"), pdf=loc["pdf_url"]))
            if len(out) >= n: break
        cursor = j.get("meta", {}).get("next_cursor")
        if not cursor: break
        time.sleep(0.2)
    return out

corpus = get_corpus(FILTER, N_PAPERS, MAILTO)
print(f"corpus: {len(corpus)} papers with a downloadable full-text PDF\n")
for i, p in enumerate(corpus, 1):
    print(f"{i:2d}. [{p['year']}] {p['venue']}  (cited {p['cited']})")
    print(f"    {p['title']}")

corpus: 20 papers with a downloadable full-text PDF

 1. [2019] Nature Communications  (cited 953)
    Zinc anode-compatible in-situ solid electrolyte interphase via cation solvation modulation
 2. [2019] Joule  (cited 936)
    A Metal-Organic Framework Host for Highly Reversible Dendrite-free Zinc Metal Anodes
 3. [2020] Nano-Micro Letters  (cited 894)
    Enhanced Potassium-Ion Storage of the 3D Carbon Superstructure by Manipulating the Nitrogen-Doped Sp
 4. [2023] Nature Communications  (cited 809)
    Revealing the closed pore formation of waste wood-derived hard carbon for advanced sodium-ion batter
 5. [2019] Nature Communications  (cited 784)
    Scalable synthesis of ant-nest-like bulk porous silicon for high-performance lithium-ion battery ano
 6. [2019] Nano-Micro Letters  (cited 771)
    Dendritic Nanostructured Waste Copper Wires for High-Energy Alkaline Battery
 7. [2019] Nature Communications  (cited 690)
    Conductive 2D metal-organic framework for high-performance cath

## 3. Download and parse the full text

Each open-access PDF is downloaded (with a normal browser header) and parsed to text. A publisher that blocks automated download (some return HTTP 403) is skipped with a note; the rest go through.

In [4]:
import fitz   # pymupdf
UA = {"User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15) "
                    "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120 Safari/537.36"}

def full_text(pdf_url):
    r = requests.get(pdf_url, headers=UA, timeout=60)
    if r.status_code != 200 or not (r.content[:4] == b"%PDF" or "pdf" in r.headers.get("content-type","")):
        return None
    doc = fitz.open(stream=r.content, filetype="pdf")
    return "\n".join(page.get_text() for page in doc)

texts = []
for p in corpus:
    try:
        t = full_text(p["pdf"])
    except Exception:
        t = None
    if t:
        texts.append({**p, "text": t})
        print(f"  ok  {len(t):6d} chars  (mAh x{t.lower().count('mah'):3d})  {p['venue'][:34]}")
    else:
        print(f"  skip (blocked/not a PDF)                    {p['venue'][:34]}")

print(f"\nfull text obtained for {len(texts)}/{len(corpus)} papers")
print("these full texts are the input to the extractor in the next part (LLM -> capacity table).")

  ok   70744 chars  (mAh x 44)  Nature Communications
  skip (blocked/not a PDF)                    Joule


  skip (blocked/not a PDF)                    Nano-Micro Letters


  ok   51068 chars  (mAh x 26)  Nature Communications


  ok   58883 chars  (mAh x 34)  Nature Communications
  skip (blocked/not a PDF)                    Nano-Micro Letters


  ok   52375 chars  (mAh x 20)  Nature Communications


...

  ok   48828 chars  (mAh x 28)  Nature Communications


  ok   65192 chars  (mAh x 20)  Nature Communications


  ok   47648 chars  (mAh x 18)  Nature Communications


  ok   61460 chars  (mAh x 18)  Nature Communications


  skip (blocked/not a PDF)                    Arrow@dit (Dublin Institute of Tec

full text obtained for 13/20 papers
these full texts are the input to the extractor in the next part (LLM -> capacity table).


## Next

Part 2 (coming next) sends each full text through the Module 2a extractor to pull a structured record (material, chemistry, specific capacity, rate, cycles) into `battery_dataset.csv`, then feeds that table into Module 1d to model it. This notebook already gives you the working harvest: **topic filter -> corpus -> parsed full text**, with no key.